Task 0: Libraries import

In [ ]:
import json
import random
import time
import pandas as pd
import requests

Task 1: Extract UPC

In [ ]:
url_1 = ""
from google.colab import userdata
API_KEY = userdata.get('MY_API_KEY')
headers_1 = {"Content-Type": "application/json",
             "ApiKey": API_KEY
}
payload_1 = {
    "Source": "TestMC",
    "Data": {
        "Request": {
            "Filters": [
                {
                    "Field": "RecModified",
                    "Operator": "Greater than or equal",
                    "Value": "2026-06-15",
                }
            ],
            "SortDescriptions": [
                {"Name": "RecModified",
                 "Direction": "Descending"}
            ],
            "Top": 100,
            "Skip": 0,
        }
    },
}
response_1 = requests.post(url_1, json=payload_1, headers=headers_1)

extracted_upcs = []

if response_1.status_code == 200:
    json_data = response_1.json()

    styles = (
        json_data.get("ApiDocument", {})
        .get("Response", {})
        .get("Styles", [])
    )

    for style in styles:
        items = style.get("Items", [])
        for item in items:
            upc = item.get("UPC")
            if upc:
                extracted_upcs.append(upc)

    print(f"Successfully extracted {len(extracted_upcs)} UPCs.")
    print("Sample extracted UPCs:", extracted_upcs[:3])
else:
    print(f"Failed. Status: {response_1.status_code}, Text: {response_1.text}")

Task 2: Build a CSV File

In [ ]:
location_pool = [1, 10, 12, 13, 19, 25, 26, 28, 3, 36, 41, 9]

data_rows = []

for upc in extracted_upcs:
    random_location = random.choice(location_pool)

    random_qty = random.randint(-10, 10)

    row = {
        "itemIdentifier": upc,
        "locationIdentifier": random_location,
        "qty": random_qty
    }
    data_rows.append(row)

df = pd.DataFrame(data_rows)

csv_filename = "inventory_data.csv"
df.to_csv(csv_filename, index=False)

print(f"CSV successfully saved as '{csv_filename}'. Total rows: {len(df)}")
print("\nPreview of the generated data:")
print(df.head())


Task 3: Extract Token

In [ ]:
token_url = ""

token_payload = {
    "client_id": API_KEY,
    "client_secret": API_KEY,
    "grant_type": "client_credentials",
    "scope": "rta"
}

headers_token = {
    "Content-Type": "application/x-www-form-urlencoded"
}

response_token = requests.post(token_url, data=token_payload, headers=headers_token)

token = None

if response_token.status_code == 200:
    token_json = response_token.json()

    token = token_json.get("access_token")

    if token:
        print("Token successfully retrieved and stored!")
    else:
        print("Response successful, but 'access_token' key was not found. Here is the response:")
        print(token_json)
else:
    print(f"Failed to retrieve token. Status Code: {response_token.status_code}")
    print(f"Error Response: {response_token.text}")

Task 4: Import External ATS Deltas

In [ ]:
if 'token' in locals() and token:
    final_url = ""

    headers_final = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }

    csv_filename = "inventory_data.csv"
    saved_df = pd.read_csv(csv_filename)

    print(f"Loaded {len(saved_df)} rows from CSV. Starting transmission...")

    for index, row in saved_df.iterrows():

        final_payload = {
            "settings": {
                "itemSetting": "UPC",
                "locationSetting": "LocationCode"
            },
            "attributes": {
                "ItemIdentifier": str(row["itemIdentifier"]),
                "LocationIdentifier": 13,
                "Qty": int(row["qty"])
            }
        }

        print(f"Sending row {index + 1}/{len(saved_df)} (UPC: {row['itemIdentifier']})...")

        response_final = requests.post(final_url, json=final_payload, headers=headers_final)

        if response_final.status_code == 200:
            print(f"-> Success: Row {index + 1} processed successfully.")

        elif response_final.status_code in [400, 444]:
            print(f"\nCRITICAL ERROR: Stopped by system on row {index + 1}.")
            print(f"Status Code: {response_final.status_code}")
            print(f"Error Message: {response_final.text}")
            break

        else:
            print(f"-> Warning: Row {index + 1} returned status {response_final.status_code}. Response: {response_final.text}")
            print("Continuing loop...")

        time.sleep(0.2)

    print("\nProcessing cycle complete.")
else:
    print("Execution aborted: No valid authentication token found from Task 3.")